# Kikuyu LLM - Step 1: BPE Tokenizer Training
**Author:** Peter Gatitu Mwangi | [github.com/bellonbits](https://github.com/bellonbits)

This notebook trains a Byte Pair Encoding (BPE) tokenizer on the Kikuyu corpus.
Run this **before** the pre-training notebook.

**Input:** `kikuyu_corpus.txt` (already in your Drive from the data pipeline)

**Output:** `kikuyu_bpe_tokenizer.json` saved to Drive

**Runtime:** No GPU needed. CPU is fine. Takes ~2-5 minutes.

## 1. Setup

In [ ]:
!pip install tokenizers -q

from tokenizers import (
    Tokenizer, models, trainers,
    pre_tokenizers, decoders, processors, normalizers
)
import os, json

print('tokenizers library ready')


tokenizers library ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ---- CONFIGURE PATHS -------------------------------------------------------
DRIVE_BASE     = '/content/drive/MyDrive/kikuyu_llm'
CORPUS_TXT     = f'{DRIVE_BASE}/kikuyu_corpus.txt'       # plain text corpus
TOKENIZER_OUT  = f'{DRIVE_BASE}/kikuyu_bpe_tokenizer.json'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Verify corpus exists
if os.path.exists(CORPUS_TXT):
    size_mb = os.path.getsize(CORPUS_TXT) / 1024 / 1024
    print(f'Corpus found: {size_mb:.2f} MB')
    # Count lines
    with open(CORPUS_TXT, encoding='utf-8') as f:
        lines = sum(1 for l in f if l.strip())
    print(f'Non-empty lines: {lines:,}')
else:
    print('ERROR: kikuyu_corpus.txt not found in Drive.')
    print('Upload it from your local kikuyu_data/ folder first.')


Corpus found: 4.91 MB
Non-empty lines: 68,121


## 2. Tokenizer Configuration

**Why BPE?** Byte Pair Encoding learns frequent character sequences from data.
Common words like `Ngai`, `Jesu`, `mwega` become single tokens.
Rare or new words are split into known subword pieces.

**Key Kikuyu note:** We use `NFC` Unicode normalization, NOT `NFKC`.
`NFKC` would decompose and strip the diacritics `i-dot`, `u-dot`, `a-tilde`
that are phonemically meaningful in Gikuyu. `NFC` preserves them.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `VOCAB_SIZE` | 32,000 | Same as LLaMA; good balance of coverage vs model size |
| `MIN_FREQUENCY` | 2 | A subword must appear at least twice to enter vocab |
| Normalizer | NFC | Preserves Kikuyu diacritics (i, u, a with marks) |
| Pre-tokenizer | Metaspace | Splits on spaces, encodes space as `_` prefix |
| Decoder | Metaspace | Reverses the above for clean text reconstruction |

In [ ]:
VOCAB_SIZE    = 32_000
MIN_FREQUENCY = 2

SPECIAL_TOKENS = [
    '<pad>',          # 0  - padding (batch alignment)
    '<unk>',          # 1  - unknown token fallback
    '<bos>',          # 2  - beginning of sequence
    '<eos>',          # 3  - end of sequence
    '<sep>',          # 4  - separator (system/user/assistant turns)
    '<|user|>',       # 5  - marks user turn in chat format
    '<|assistant|>',  # 6  - marks assistant turn in chat format
    '<|system|>',     # 7  - marks system prompt in chat format
]

print(f'Vocab size    : {VOCAB_SIZE:,}')
print(f'Min frequency : {MIN_FREQUENCY}')
print(f'Special tokens: {len(SPECIAL_TOKENS)}')
for i, tok in enumerate(SPECIAL_TOKENS):
    print(f'  [{i}] {tok}')


Vocab size    : 32,000
Min frequency : 2
Special tokens: 8
  [0] <pad>
  [1] <unk>
  [2] <bos>
  [3] <eos>
  [4] <sep>
  [5] <|user|>
  [6] <|assistant|>
  [7] <|system|>


## 3. Build & Train the Tokenizer

The BPE trainer reads the full corpus and iteratively merges the most
frequent adjacent character pairs until reaching `VOCAB_SIZE` merges.

This takes 2-5 minutes on CPU depending on corpus size.

In [ ]:
# Step 1: Create BPE model
tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))

# Step 2: NFC normalization - CRITICAL for Kikuyu diacritics
# NFC preserves i-with-dot, u-with-dot, a-with-tilde as single code points
tokenizer.normalizer = normalizers.NFC()

# Step 3: Metaspace pre-tokenizer
# Splits text on whitespace, marks word starts with special _ prefix
# 'mwega sana' -> ['_mwega', '_sana'] -> cleaner reconstruction
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

# Step 4: Metaspace decoder (reverses pre-tokenizer for clean output)
tokenizer.decoder = decoders.Metaspace()

# Step 5: Configure trainer
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
    initial_alphabet=[],  # learn alphabet from data
)

print('Tokenizer configured. Starting training...')
print('(This takes 2-5 minutes on CPU)')


Tokenizer configured. Starting training...
(This takes 2-5 minutes on CPU)


In [ ]:
# Train on the corpus text file
tokenizer.train([CORPUS_TXT], trainer)

actual_vocab = tokenizer.get_vocab_size()
print(f'Training complete!')
print(f'Vocabulary size: {actual_vocab:,}')


Training complete!
Vocabulary size: 32,000


## 4. Configure Post-Processing

This step tells the tokenizer to automatically wrap every encoded sequence
with `<bos>` at the start and `<eos>` at the end.

Without this, `<bos>` and `<eos>` exist in the vocabulary but never appear
in outputs unless you add them manually every time.

The `pair` template is used for two-segment inputs (e.g. question + answer).

In [ ]:
bos_id = tokenizer.token_to_id('<bos>')
eos_id = tokenizer.token_to_id('<eos>')
sep_id = tokenizer.token_to_id('<sep>')

print(f'<bos> id : {bos_id}')
print(f'<eos> id : {eos_id}')
print(f'<sep> id : {sep_id}')

tokenizer.post_processor = processors.TemplateProcessing(
    single=f'<bos>:0 $A:0 <eos>:0',
    pair=f'<bos>:0 $A:0 <sep>:0 $B:1 <eos>:1',
    special_tokens=[
        ('<bos>', bos_id),
        ('<eos>', eos_id),
        ('<sep>', sep_id),
    ],
)

print('Post-processor configured (auto-adds <bos> and <eos> to every sequence)')


<bos> id : 2
<eos> id : 3
<sep> id : 4
Post-processor configured (auto-adds <bos> and <eos> to every sequence)


## 5. Test the Tokenizer

Three tests:
1. **Roundtrip** - encode then decode must return the original text exactly
2. **Special tokens** - verify `<bos>` and `<eos>` are automatically added
3. **Fertility score** - average tokens per word (target: under 2.5 for Kikuyu)

In [ ]:
test_sentences = [
    'Ngai niombire iguru na thi',
    'Muthenya wa mugwanja ni muthenya wa guhuruka',
    'Jesu niaciarirwo ni Mariamu',
    'Wi mwega na uhoro wa Ngai',
    'Agikuyu ni anduu a mithemba ya Bantu',
    'Kirinyaga ni muima murimu wa Kenya',
]

print('=' * 65)
print('TOKENIZER TEST RESULTS')
print('=' * 65)

all_ok = True
for sentence in test_sentences:
    encoded  = tokenizer.encode(sentence)
    decoded  = tokenizer.decode(encoded.ids)
    roundtrip_ok = sentence.strip() in decoded or decoded.strip() == sentence.strip()
    if not roundtrip_ok:
        all_ok = False

    print(f'Input    : {sentence}')
    print(f'Tokens   : {encoded.tokens}')
    print(f'IDs      : {encoded.ids}')
    print(f'Decoded  : {decoded}')
    print(f'N tokens : {len(encoded.ids)}  |  Roundtrip: {"OK" if roundtrip_ok else "FAIL"}')
    print('-' * 65)

print(f'All roundtrips passed: {all_ok}')


TOKENIZER TEST RESULTS
Input    : Ngai niombire iguru na thi
Tokens   : ['<bos>', '▁Ngai', '▁n', 'io', 'mbire', '▁igu', 'ru', '▁na', '▁thi', '<eos>']
IDs      : [2, 221, 83, 5872, 2199, 7670, 142, 89, 658, 3]
Decoded  : Ngai niombire iguru na thi
N tokens : 10  |  Roundtrip: OK
-----------------------------------------------------------------
Input    : Muthenya wa mugwanja ni muthenya wa guhuruka
Tokens   : ['<bos>', '▁Mu', 'thenya', '▁wa', '▁mu', 'gwa', 'nja', '▁ni', '▁mu', 'thenya', '▁wa', '▁gu', 'huru', 'ka', '<eos>']
IDs      : [2, 561, 361, 98, 4238, 271, 363, 7647, 4238, 361, 98, 13313, 1926, 105, 3]
Decoded  : Muthenya wa mugwanja ni muthenya wa guhuruka
N tokens : 15  |  Roundtrip: OK
-----------------------------------------------------------------
Input    : Jesu niaciarirwo ni Mariamu
Tokens   : ['<bos>', '▁Jes', 'u', '▁n', 'ia', 'ciarirwo', '▁ni', '▁Mariamu', '<eos>']
IDs      : [2, 381, 71, 83, 123, 5684, 7647, 3390, 3]
Decoded  : Jesu niaciarirwo ni Mariamu
N tokens : 9 

In [ ]:
# Fertility score: average tokens per word
# Measures how efficiently the tokenizer represents the language
# - Fertility = 1.0 : every word is one token (ideal but unrealistic)
# - Fertility < 2.0 : very good for Kikuyu (morphologically rich)
# - Fertility > 3.0 : too much splitting, tokenizer needs improvement

print('Calculating fertility score on corpus...')
total_words  = 0
total_tokens = 0
sample_size  = 5000  # sample first 5000 lines for speed

with open(CORPUS_TXT, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= sample_size or not line.strip():
            continue
        words  = line.strip().split()
        enc    = tokenizer.encode(line.strip())
        # subtract <bos> and <eos> added by post-processor
        n_toks = len(enc.ids) - 2
        total_words  += len(words)
        total_tokens += n_toks

fertility = total_tokens / max(total_words, 1)
print(f'Fertility score : {fertility:.2f} tokens/word')
print(f'Words sampled   : {total_words:,}')
print(f'Tokens produced : {total_tokens:,}')

if fertility < 2.0:
    print('Excellent - tokenizer is well-optimised for Kikuyu')
elif fertility < 2.5:
    print('Good - acceptable for Kikuyu morphology')
elif fertility < 3.0:
    print('Fair - consider larger vocab or more corpus data')
else:
    print('High fertility - tokenizer may need improvement')


Calculating fertility score on corpus...
Fertility score : 1.12 tokens/word
Words sampled   : 61,781
Tokens produced : 69,491
Excellent - tokenizer is well-optimised for Kikuyu


## 6. Inspect the Vocabulary

In [ ]:
vocab = tokenizer.get_vocab()  # {token: id}
id_to_token = {v: k for k, v in vocab.items()}

# Show special tokens
print('Special tokens:')
for tok in SPECIAL_TOKENS:
    print(f'  {tok:<20} id={tokenizer.token_to_id(tok)}')

# Show sample of learned subwords
print(f'\nSample learned subwords (IDs 8-50):')
sample_ids = list(range(8, 51))
tokens_sample = [id_to_token.get(i, '?') for i in sample_ids]
print(' '.join(tokens_sample))

# Count token length distribution
lengths = [len(tok) for tok in vocab.keys() if not tok.startswith('<')]
print(f'\nToken length distribution:')
for l in range(1, 10):
    count = sum(1 for x in lengths if x == l)
    bar   = '#' * (count // 100)
    print(f'  len={l}: {count:>5}  {bar}')

# Show some common Kikuyu words that should be single tokens
print('\nKikuyu keyword coverage (should be single tokens):')
keywords = ['Ngai', 'Jesu', 'mwega', 'mundu', 'thi', 'muciia',
            'uhoro', 'Kirinyaga', 'Agikuyu', 'muthenya']
for w in keywords:
    tok_id = tokenizer.token_to_id(f'_{w}') or tokenizer.token_to_id(w)
    enc    = tokenizer.encode(w).tokens
    n      = len([t for t in enc if t not in ('<bos>', '<eos>')])
    status = 'single token' if n == 1 else f'{n} tokens: {enc}'
    print(f'  {w:<15} {status}')


Special tokens:
  <pad>                id=0
  <unk>                id=1
  <bos>                id=2
  <eos>                id=3
  <sep>                id=4
  <|user|>             id=5
  <|assistant|>        id=6
  <|system|>           id=7

Sample learned subwords (IDs 8-50):

 ! ( ) , - . 0 1 2 3 4 5 6 7 8 9 : ; ? A B C D E F G H I J K L M N O P R S T U V W Y

Token length distribution:
  len=1:    75  
  len=2:   349  ###
  len=3:  1011  ##########
  len=4:  2129  #####################
  len=5:  3545  ###################################
  len=6:  4383  ###########################################
  len=7:  4952  #################################################
  len=8:  4496  ############################################
  len=9:  3542  ###################################

Kikuyu keyword coverage (should be single tokens):
  Ngai            single token
  Jesu            2 tokens: ['<bos>', '▁Jes', 'u', '<eos>']
  mwega           single token
  mundu           2 tokens: ['<bos>', '▁mu

## 7. Save Tokenizer to Drive

In [ ]:
tokenizer.save(TOKENIZER_OUT)

size_kb = os.path.getsize(TOKENIZER_OUT) / 1024
print(f'Tokenizer saved: {TOKENIZER_OUT}')
print(f'File size      : {size_kb:.0f} KB')

# Reload and verify it works
from tokenizers import Tokenizer as T2
reloaded = T2.from_file(TOKENIZER_OUT)
test_enc = reloaded.encode('Ngai niombire iguru na thi')
print(f'\nVerification reload:')
print(f'  Input  : Ngai niombire iguru na thi')
print(f'  Tokens : {test_enc.tokens}')
print(f'  Reload : OK')


Tokenizer saved: /content/drive/MyDrive/kikuyu_llm/kikuyu_bpe_tokenizer.json
File size      : 2405 KB

Verification reload:
  Input  : Ngai niombire iguru na thi
  Tokens : ['<bos>', '▁Ngai', '▁n', 'io', 'mbire', '▁igu', 'ru', '▁na', '▁thi', '<eos>']
  Reload : OK


## 8. Summary & Next Step

In [ ]:
print('=' * 55)
print('TOKENIZER TRAINING COMPLETE')
print('=' * 55)
print(f'  Vocabulary size : {tokenizer.get_vocab_size():,}')
print(f'  Fertility score : {fertility:.2f} tokens/word')
print(f'  Saved to        : {TOKENIZER_OUT}')
print()
print('Files now in your Drive:')
for fname in ['kikuyu_corpus.txt', 'kikuyu_corpus.jsonl', 'kikuyu_bpe_tokenizer.json']:
    fpath = f'{DRIVE_BASE}/{fname}'
    if os.path.exists(fpath):
        sz = os.path.getsize(fpath) / 1024 / 1024
        print(f'  {fname:<40} {sz:.1f} MB  OK')
    else:
        print(f'  {fname:<40} MISSING')
print()
print('Next step: open kikuyu_pretrain.ipynb and run all cells.')
print('The pre-training notebook will now find kikuyu_bpe_tokenizer.json.')


TOKENIZER TRAINING COMPLETE
  Vocabulary size : 32,000
  Fertility score : 1.12 tokens/word
  Saved to        : /content/drive/MyDrive/kikuyu_llm/kikuyu_bpe_tokenizer.json

Files now in your Drive:
  kikuyu_corpus.txt                        4.9 MB  OK
  kikuyu_corpus.jsonl                      5.0 MB  OK
  kikuyu_bpe_tokenizer.json                2.3 MB  OK

Next step: open kikuyu_pretrain.ipynb and run all cells.
The pre-training notebook will now find kikuyu_bpe_tokenizer.json.
